# Projeto 01 - Xadrez

Este notebook implementa uma cena 3D interativa de um tabuleiro de xadrez, renderizada
com **OpenGL** (via PyOpenGL) e gerenciamento de janela/eventos com **GLFW**.

O tabuleiro e as peças (peão, torre, cavalo, bispo, rei e rainha) são construídos
manualmente a partir de primitivas geométricas simples (paralelepípedos, prismas triangulares e
troncos de pirâmide/trapézios), combinando vários "blocos" para formar cada peça.

O usuário pode interagir com o tabuleiro através dos seguintes comandos:

`WASD - Movimentação de um peão branco`

`Enter - Captura a peça que está em cima`

`CV - Rotação da torre branca à esquerda`

`QE - Rotação horizontal do tabuleiro`

`RF - Rotação vertical do tabuleiro`

`ZX - Zoom do tabuleiro`

`B - Rage Quit`

`P - Ativa/Desativa a visualização poligonal`

`Espaço - Reseta as condições iniciais`

Integrantes:
```
Luís Henrique de Queiroz Veras -- 14592414
Vinícius de Sá Ferreira        -- 15491650
```

### Importando bibliotecas

Instalação (opcional) e importação das bibliotecas usadas no projeto:

- **PyOpenGL**: bindings Python para a API OpenGL, usada para desenhar a cena 3D.
- **glfw**: cria a janela, o contexto OpenGL e captura eventos de teclado.
- **numpy**: armazena e manipula os vértices e as matrizes de transformação (4x4).
- **PyGLM**: biblioteca de matemática para gráficos (importada como `glm`), utilizada
  aqui apenas de forma auxiliar — as transformações 3D principais são implementadas
  manualmente com `numpy` mais adiante.

In [1]:
# Instala as dependências necessárias para renderização 3D (executar apenas uma vez,
# se ainda não estiverem instaladas no ambiente).
#!pip install PyOpenGL PyOpenGL_accelerate glfw PyGLM numpy

In [2]:
from OpenGL.GL import *       # Funções e constantes principais da API OpenGL (glCreateShader, glDrawArrays, etc.)
import OpenGL.GL.shaders      # Utilitários para compilação/linkagem de shaders (não usado diretamente, mas mantido como referência)
import numpy as np            # Arrays para vértices e matrizes de transformação
import glm                    # Biblioteca de matemática gráfica (não utilizada diretamente no restante do código)
import glfw                   # Criação de janela, contexto OpenGL e tratamento de eventos de teclado

### Inicializando a cena

Nesta etapa são feitos os passos básicos para se ter uma cena OpenGL funcional:

1. Cria a janela e o contexto OpenGL com o **GLFW**.
2. Define o código-fonte dos **shaders** (vertex e fragment shader), escritos em GLSL.
3. Compila cada shader e verifica se houve erro de compilação.
4. Faz o **link** dos shaders em um único *program* na GPU.
5. Recupera as *locations* (endereços) dos atributos/uniforms do shader, que serão
   usados mais adiante para enviar dados (posição dos vértices, cor e matriz de
   transformação) da CPU para a GPU.

In [3]:
# Cria a janela
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.TRUE);           # A janela já começa visível (mesmo antes do loop principal)
window = glfw.create_window(800, 800, "Xadrez", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()

# Torna o contexto OpenGL da janela criada o contexto "atual" para esta thread,
# permitindo que os comandos OpenGL seguintes sejam executados sobre ele.
glfw.make_context_current(window)

# Shaders em 3D
# Vertex shader: roda uma vez por vértice. Recebe a posição do vértice (`position`)
# e aplica a matriz de transformação (modelo + câmera) enviada via uniform.
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

# Fragment shader: roda uma vez por pixel/fragmento. Apenas pinta o fragmento com a
# cor recebida via uniform (cor sólida, sem iluminação).
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

# Aloca o programa (program) e os objetos de shader (vertex e fragment) na GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Associa o código-fonte GLSL a cada objeto de shader criado
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compila o vertex shader e verifica se a compilação foi bem-sucedida
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

# Compila o fragment shader e verifica se a compilação foi bem-sucedida
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Linkagem: anexa os dois shaders compilados ao programa e faz o link,
# gerando o pipeline programável final que será usado para desenhar
glAttachShader(program, vertex)
glAttachShader(program, fragment)

glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

# Ativa o programa de shaders para ser usado nas próximas chamadas de desenho
glUseProgram(program)

# Recupera as "locations" (endereços) do atributo de posição e dos uniforms de cor e
# matriz de transformação, para que possamos enviar esses dados a cada frame/objeto
loc = glGetAttribLocation(program, "position")
loc_color = glGetUniformLocation(program, "color")
loc_transformation = glGetUniformLocation(program, "mat_transformation")

### Objetos

Aqui são definidas as **primitivas geométricas** usadas para montar cada peça de
xadrez e o tabuleiro, todas construídas manualmente como listas de triângulos
(3 vértices por triângulo, sentido anti-horário quando visto de fora):

- `cubo(c, lx, ly, lz)`: um paralelepípedo centrado em `c`, com dimensões `lx`, `ly`
  e `lz` (largura, altura e profundidade). Usado para bases, corpos e detalhes das peças.
- `prismaTri(c, lx, ly, lz)`: um prisma de base triangular, usado por exemplo nas
  "orelhas" do cavalo e no topo do bispo.
- `trapz(c, l_m, l_M, h)`: um tronco de pirâmide (base menor `l_m`, base maior `l_M`
  e altura `h`), usado nas partes cônicas das peças (torre, rei, rainha).

Também é definida a classe `Objeto`, que guarda os metadados de cada peça/parte
desenhada (onde seus vértices começam no buffer, quantos vértices ela tem, sua cor,
sua posição no tabuleiro e se ela ainda está "ativa", ou seja, se ainda não foi
capturada).

In [4]:
vertices = []

# Formas Geométricas
# Cada função abaixo GERA os vértices de uma forma 3D e os ACRESCENTA (append) na
# lista global `vertices`. Os vértices são agrupados de 3 em 3, formando triângulos
# (2 triângulos por face, já que a GPU só desenha triângulos).

def cubo(c, lx, ly, lz):
    """Gera um paralelepípedo (cubo/caixa) centrado no ponto `c`.

    Parâmetros:
        c: centro do cubo (x, y, z)
        lx, ly, lz: largura total nos eixos X, Y e Z (a função divide por 2
                    internamente para obter a "meia-largura" de cada lado)
    """
    global vertices

    lx /= 2
    ly /= 2
    lz /= 2

    vertices += [
        # Bottom (face de baixo, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),

        # Top (face de cima, 2 triângulos)
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        # Front (face da frente, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Back (face de trás, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        # Left (face esquerda, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),

        # Right (face direita, 2 triângulos)
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz)
    ]

def prismaTri(c, lx, ly, lz):
    """Gera um prisma de base triangular centrado no ponto `c`.

    Usado para detalhes pontiagudos das peças (ex.: "orelhas" do cavalo, topo do
    bispo). `lx`, `ly` e `lz` funcionam como escalas/dimensões do prisma nos três
    eixos (podem ser negativas para espelhar o prisma).
    """
    global vertices

    lx /= 2
    ly /= 2
    lz /= 2

    vertices += [
        # Bottom Tri (triângulo da base)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),

        # Top Tri (triângulo do topo)
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Back Rect (face retangular de trás, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Left Rect (face retangular esquerda, 2 triângulos)
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),

        # Front Rect (face retangular inclinada da frente, a "hipotenusa" do prisma)
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz)
    ]

def trapz(c, l_m, l_M, h):
    """Gera um tronco de pirâmide (formato de "vaso"/cone truncado de base quadrada),
    centrado em `c`.

    Parâmetros:
        c: centro do sólido (x, y, z)
        l_m: lado da base MENOR (embaixo)
        l_M: lado da base MAIOR (em cima)
        h: altura total (dividida por 2 internamente, como nas outras funções)

    Usado nas partes "cônicas" das peças, como o corpo da torre, do rei e da rainha.
    """
    global vertices

    l_m /= 2
    l_M /= 2
    h /= 2

    vertices += [
        # Bottom (base menor, embaixo)
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),

        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_m, c[1]-h, c[2]+l_m),

        # Top (base maior, em cima)
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]-l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),

        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_M, c[1]+h, c[2]-l_M),

        # Front (face lateral da frente, ligando base menor à maior)
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_m, c[1]-h, c[2]-l_m),

        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]-l_M),

        # Back (face lateral de trás)
        (c[0]-l_m, c[1]-h, c[2]+l_m),
        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        (c[0]+l_m, c[1]-h, c[2]+l_m),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        # Left (face lateral esquerda)
        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_m, c[1]-h, c[2]+l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),

        (c[0]-l_m, c[1]-h, c[2]-l_m),
        (c[0]-l_M, c[1]+h, c[2]+l_M),
        (c[0]-l_M, c[1]+h, c[2]-l_M),

        # Right (face lateral direita)
        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_M, c[1]+h, c[2]-l_M),
        (c[0]+l_M, c[1]+h, c[2]+l_M),

        (c[0]+l_m, c[1]-h, c[2]-l_m),
        (c[0]+l_M, c[1]+h, c[2]+l_M),
        (c[0]+l_m, c[1]-h, c[2]+l_m)
    ]

class Objeto:
    """Representa um objeto desenhável na cena (uma peça inteira ou um pedaço do
    tabuleiro).

    Atributos:
        id: identificador textual do objeto (ex.: "peao_branco", "torre_branca",
            "tabuleiro"). Usado no loop principal para aplicar comportamentos
            específicos a cada tipo de objeto.
        inicio: índice, dentro do array `vertices`, onde os vértices deste objeto
                começam.
        quant: quantidade de vértices que pertencem a este objeto (usado por
               `glDrawArrays` para saber quantos vértices desenhar a partir de
               `inicio`).
        cor: cor RGB do objeto (lista/tupla de 3 valores entre 0 e 1).
        pos: posição (coluna, linha) do objeto no grid do tabuleiro (8x8).
        ativa: indica se o objeto ainda deve ser desenhado (False após ser
               "capturado").
    """
    def __init__(self, id, inicio, quant, cor=[0,0,0], pos=[0,0]):
        self.id = id
        self.inicio = inicio
        self.quant = quant
        self.cor = cor
        self.pos = pos
        self.ativa = True 

objetos = []  # Lista com todos os objetos (peças + partes do tabuleiro) da cena

In [5]:
# Paleta de cores usada na cena
COR = {
    "fundo": [0.784, 0.949, 0.933],        # cor de fundo (background) da janela
    "tabuleiro": {
        "base": [0.588, 0.392, 0.212],     # cor da base/moldura do tabuleiro
        "brancas": [0.868, 0.879, 0.77],   # cor das casas "brancas"
        "pretas": [0.463, 0.584, 0.341]    # cor das casas "pretas"
    },
    "peça": {
        0: [1, 1, 1],       # peças do lado 0 (brancas)
        1: [0.1, 0.1, 0.1]  # peças do lado 1 (pretas)
    }
}

# Objetos (0 - Brancas / 1 - Pretas)
# Cada função `add_<peça>` monta a geometria de uma peça de xadrez combinando várias
# chamadas a `cubo`, `prismaTri` e `trapz` (empilhadas verticalmente a partir da
# casa do tabuleiro em que a peça está), e registra o resultado como um novo
# `Objeto` na lista `objetos`.

def add_peao(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)          # base do peão
    cubo(grid[pos]+[0,0.054,0], ds/4, 0.018, ds/4)           # corpo/pescoço
    cubo(grid[pos]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5) # "cabeça" do peão
    quant = len(vertices)-inicio
    # O peão em (3,1) é o peão controlável pelo jogador e não recebe sufixo de cor
    # no id, pois é tratado de forma especial no loop principal (movimentação livre)
    id = "" if pos[0]==3 and pos[1]==1 else ("_branco" if side==0 else "_preto")
    objetos.append(Objeto("peao"+id, inicio, quant, COR["peça"][side], pos))

def add_cavalo(vertices, objetos, side, pos):
    dir = -1 if side==0 else 1  # inverte a orientação da "cabeça" do cavalo conforme o lado
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)                  # base
    cubo(grid[pos]+[0,0.054,dir*ds/12], ds/3.5, 0.018, ds/3.5)       # pescoço, levemente deslocado
    cubo(grid[pos]+[0,0.073,0], ds/3.5, 0.02, ds/2.2)                # "cabeça" alongada
    prismaTri(grid[pos]+[dir*ds/10,0.088,dir*ds/10], ds/10, 0.01, ds/10)   # orelha 1
    prismaTri(grid[pos]+[-dir*ds/10,0.088,dir*ds/10], -ds/10, -0.01, ds/10) # orelha 2 (espelhada)
    quant = len(vertices)-inicio
    objetos.append(Objeto("cavalo_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_torre(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0, 0.0425, 0], ds/1.8, 0.005, ds/1.8)          # base
    trapz(grid[pos]+[0, 0.065, 0], ds/2.0, ds/3.5, 0.04)           # corpo cônico
    cubo(grid[pos]+[0, 0.088, 0], ds/2.2, 0.006, ds/2.2)           # topo plano
    offset = ds/6.5
    y_ameia = 0.094
    # As 4 "ameias" (dentes) no topo da torre, um em cada canto
    cubo(grid[pos]+[-offset, y_ameia, -offset], ds/7, 0.008, ds/7)
    cubo(grid[pos]+[-offset, y_ameia,  offset], ds/7, 0.008, ds/7)
    cubo(grid[pos]+[ offset, y_ameia, -offset], ds/7, 0.008, ds/7)
    cubo(grid[pos]+[ offset, y_ameia,  offset], ds/7, 0.008, ds/7)
    quant = len(vertices)-inicio
    objetos.append(Objeto("torre_"+("branca" if side==0 else "preta"), inicio, quant, COR["peça"][side], pos))
    
def add_bispo(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)                       # base
    cubo(grid[pos]+[0,0.055,0], ds/4, 0.02, ds/4)                         # corpo inferior
    cubo(grid[pos]+[0,0.0685,0], ds/3+0.004, 0.007, ds/3+0.004)           # "colarinho"
    # Dois prismas triangulares espelhados formam o "capuz" pontudo do bispo
    prismaTri(grid[pos]+[0.002,0.065+ds/6,0], -ds/3, ds/3, -ds/3)
    prismaTri(grid[pos]+[-0.002,0.065+ds/6,0], ds/3, ds/3, ds/3)
    cubo(grid[pos]+[0,0.065+19*ds/48,0], ds/8, ds/8, ds/8)                # bolinha do topo
    quant = len(vertices)-inicio
    objetos.append(Objeto("bispo_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_rei(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)      # base
    cubo(grid[pos]+[0,0.063,0], ds/4, 0.036, ds/4)       # corpo
    trapz(grid[pos]+[0,0.086,0], ds/4, ds/2, 0.01)       # "colarinho" cônico, alargando
    trapz(grid[pos]+[0,0.0935,0], ds/2, ds/8, 0.005)     # afunilando de novo até o topo
    cubo(grid[pos]+[0,0.1035,0], ds/8, 0.015, ds/8)      # base da cruz
    cubo(grid[pos]+[0,0.1035,0], 0.013, ds/8, ds/8)      # haste da cruz (cruzada com a anterior)
    quant = len(vertices)-inicio
    objetos.append(Objeto("rei_"+("branco" if side==0 else "preto"), inicio, quant, COR["peça"][side], pos))

def add_rainha(vertices, objetos, side, pos):
    inicio = len(vertices)
    cubo(grid[pos]+[0,0.0425,0], ds/2, 0.005, ds/2)     # base
    cubo(grid[pos]+[0,0.063,0], ds/4, 0.036, ds/4)      # corpo
    trapz(grid[pos]+[0,0.086,0], ds/4, ds/2, 0.01)      # "coroa" cônica, alargando para cima
    cubo(grid[pos]+[0,0.091+ds/16,0], ds/8, ds/8, ds/8) # bolinha do topo
    quant = len(vertices)-inicio
    objetos.append(Objeto("rainha_"+("branca" if side==0 else "preta"), inicio, quant, COR["peça"][side], pos))

# Tabuleiro
# `grid` guarda a posição (x, y, z) no mundo 3D correspondente a cada casa (i, j)
# do tabuleiro 8x8, para que as peças possam ser posicionadas em cima da casa certa.
grid = np.zeros((8, 8, 3), dtype=np.float32)
ds = 0.4/8  # tamanho (lado) de cada casa do tabuleiro
for i in range(8):
    for j in range(8):
        x = -0.2 + ds/2 + i*ds
        z = -0.2 + ds/2 + j*ds
        grid[i,j] = [x, -0.15, z]

        # Desenha a casa (i, j) como um cubo achatado, alternando as cores
        # (padrão xadrez) de acordo com a paridade de i+j
        inicio = len(vertices)
        cubo([x, -0.15, z], ds, 0.08, ds)
        quant = len(vertices)-inicio
        if (i + j) % 2 != 0:
            objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["brancas"]))
        else:
            objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["pretas"]))

# Base/moldura sólida por baixo de todas as casas do tabuleiro
inicio = len(vertices)
cubo([0,-0.16,0], 0.5, 0.08, 0.5)
quant = len(vertices)-inicio
objetos.append(Objeto("tabuleiro", inicio, quant, COR["tabuleiro"]["base"]))

# Peões: uma fileira inteira (i de 0 a 7) para cada lado, nas linhas 1 (brancas) e 6 (pretas)
for i in range(8):
    add_peao(vertices, objetos, 0, (i,1))
    add_peao(vertices, objetos, 1, (i,6))

# Torres: cantos do tabuleiro (colunas 0 e 7), linhas 0 (brancas) e 7 (pretas)
for i in range(2):
    add_torre(vertices, objetos, 0, (7*i,0))
    add_torre(vertices, objetos, 1, (7*i,7))

# Cavalos: colunas 1 e 6
for i in range(2):
    add_cavalo(vertices, objetos, 0, (1+5*i,0))
    add_cavalo(vertices, objetos, 1, (1+5*i,7))

# Bispos: colunas 2 e 5
for i in range(2):
    add_bispo(vertices, objetos, 0, (2+3*i,0))
    add_bispo(vertices, objetos, 1, (2+3*i,7))

# Reis: coluna 3
add_rei(vertices, objetos, 0, (3,0))
add_rei(vertices, objetos, 1, (3,7))


# Rainhas: coluna 4
add_rainha(vertices, objetos, 0, (4,0))
add_rainha(vertices, objetos, 1, (4,7))


# Converte a lista de tuplas de vértices em um array numpy de ponto flutuante,
# formato exigido pelo OpenGL para popular o buffer de vértices (VBO)
vertices = np.array(vertices, dtype=np.float32)

### Configuração do buffer

Aqui os vértices (já calculados na etapa anterior) são enviados para a GPU através de
um **VBO** (*Vertex Buffer Object*), e o atributo `position` do vertex shader é
configurado para ler esses dados corretamente (3 floats por vértice, sem
espaçamento extra entre eles).

In [6]:
# Gera um identificador de buffer e o define como o buffer de vértices ativo
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Envia todos os vértices calculados para a memória da GPU.
# GL_DYNAMIC_DRAW indica que o conteúdo pode ser alterado com frequência
# (aqui, na prática, os vértices não mudam após esta chamada, mas o uso deste modo
# é seguro e permitiria atualizações futuras do buffer).
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)

# Habilita o atributo "position" do vertex shader para leitura a partir do buffer
glEnableVertexAttribArray(loc)

# Descreve o layout dos dados no buffer: `stride` é o espaço (em bytes) entre o
# início de um vértice e o início do próximo, e `offset` é o deslocamento inicial
# dentro do buffer.
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Loop principal

Esta última parte do notebook contém:

- As **variáveis de estado** da aplicação (posição do peão, ângulos de rotação da
  câmera e da torre, zoom, flags de captura e "rage quit", etc.);
- A função `key_event`, callback chamada pelo GLFW sempre que uma tecla é
  pressionada, mantida pressionada (`REPEAT`) ou solta (`RELEASE`), que atualiza
  essas variáveis de estado de acordo com a tecla pressionada;
- As funções auxiliares de **matrizes de transformação 3D** (`translacao`,
  `rotacao`, `escala` e `multiplica_matriz`), usadas para posicionar, girar e
  escalar cada objeto antes de desenhá-lo;
- O **loop de renderização** propriamente dito, que a cada frame recalcula a matriz
  de transformação de cada objeto da cena (peão em movimento, torre girando, zoom
  na peça-alvo, animação de "rage quit", rotação/zoom global da câmera) e emite os
  comandos de desenho (`glDrawArrays`) para a GPU.

In [7]:
dt = 0.05  # "passo" usado para incrementar ângulos/escala a cada tecla pressionada

# Variáveis para translação do peão (posição em casas do grid, não em coordenadas do mundo)
peao_x = 3
peao_z = 1

# Variável para rotação da câmera (ângulos acumulados nos eixos X e Y)
a_x = a_y = 0

# Variável para rotação da torre da esquerda (ângulo acumulado em torno do eixo Y)
a_torre = 0
va_torre = 0  # velocidade angular atual da torre (0, +dt ou -dt)

# Variável para escala uniforme (zoom da câmera, somado a um valor base fixo depois)
sc = 0


# Variáveis de velocidade angular da câmera (incrementadas/zeradas conforme teclas
# são pressionadas/soltas, dando o efeito de rotação contínua enquanto a tecla é mantida)
va_x = va_y = 0

# Variável de velocidade da escala uniforme (zoom contínuo enquanto a tecla é mantida)
vsc = 0

# Variável para alternar entre visualização sólida (preenchida) e "wireframe" (poligonal)
poly = False

capturar = False    # flag: True por um frame quando o jogador aperta Enter para capturar uma peça
rage_quit = False    # flag: True quando a animação de "rage quit" está em andamento
tempo_rage = 0.0     # cronômetro (em "unidades de dt") usado para animar o rage quit

# COMANDOS:
#   WASD - Movimentação do peão branco
#   Enter - Captura a peça que está em cima
#
#   QE - Rotação horizontal do tabuleiro
#   RF - Rotação vertical do tabuleiro
#   ZX - Zoom do tabuleiro
#
#   CV - Rotação da torre
#
#   B - Rage Quit
#   P - Ativa/Desativa a visualização poligonal
#   Espaço - Reseta as condições iniciais

def key_event(window,key,scancode,action,mods):
    """Callback de teclado do GLFW.

    Chamada automaticamente pelo GLFW sempre que o estado de uma tecla muda.
    `action` indica se a tecla foi pressionada (PRESS), está sendo mantida
    pressionada (REPEAT) ou foi solta (RELEASE).
    """
    global peao_x, peao_z
    global a_x, a_y, va_x, va_y
    global sc, vsc
    global poly
    global a_torre, va_torre
    global capturar, rage_quit, tempo_rage

    if action == glfw.PRESS or action == glfw.REPEAT:
        # Movimentação do peão (WASD), limitada aos limites do tabuleiro (0 a 7)
        if key == glfw.KEY_W and peao_z < 7: peao_z += 1
        if key == glfw.KEY_S and peao_z > 0: peao_z -= 1
        if key == glfw.KEY_A and peao_x > 0: peao_x -= 1
        if key == glfw.KEY_D and peao_x < 7: peao_x += 1

        # Rotação vertical da câmera (R/F)
        if key == glfw.KEY_R: va_x = dt
        if key == glfw.KEY_F: va_x = -dt

        # Rotação horizontal da câmera (Q/E)
        if key == glfw.KEY_E: va_y = dt
        if key == glfw.KEY_Q: va_y = -dt

        # Zoom da câmera (Z/X)
        if key == glfw.KEY_X: vsc = dt
        if key == glfw.KEY_Z: vsc = -dt

        # Rotação manual da torre branca da esquerda (C/V)
        if key == glfw.KEY_C: va_torre = -dt
        if key == glfw.KEY_V: va_torre = dt

        # Captura a peça que estiver na mesma casa do peão (Enter)
        if key == glfw.KEY_ENTER: capturar = True

        # Aciona a animação de "rage quit" (B)
        if key == glfw.KEY_B: rage_quit = True 

        # Alterna entre modo sólido e modo wireframe (P)
        if key == glfw.KEY_P: poly = not poly

        # Reseta todo o estado da cena para a condição inicial (Espaço)
        if key == glfw.KEY_SPACE:
            peao_x = 3
            peao_z = 1
            a_x = a_y = 0
            sc = 0
            a_torre = 0
            poly = False
            rage_quit = False
            tempo_rage = 0.0 
            for obj in objetos:
                obj.ativa = True

    elif action == glfw.RELEASE:
        # Ao soltar as teclas de movimento contínuo, zera a respectiva velocidade
        # para que o movimento pare (evita que a câmera/torre continue girando
        # infinitamente após soltar a tecla)
        if key in (glfw.KEY_R, glfw.KEY_F): va_x = 0
        if key in (glfw.KEY_Q, glfw.KEY_E): va_y = 0
        if key in (glfw.KEY_Z, glfw.KEY_X): vsc = 0
        if key in (glfw.KEY_C, glfw.KEY_V): va_torre = 0


# Registra `key_event` como a função a ser chamada pelo GLFW a cada evento de teclado
glfw.set_key_callback(window,key_event)

In [8]:
# Matrizes de transformação 3D
#
# As matrizes abaixo são todas 4x4, representadas como arrays numpy "achatados"
# (shape (1,16), ordem linha-a-linha / row-major), pois é assim que o array final é
# enviado para o uniform `mat_transformation` do vertex shader via glUniformMatrix4fv
# (com o parâmetro `transpose=GL_TRUE` no ponto de uso, tratando a conversão para o
# formato column-major esperado internamente pelo OpenGL).

def multiplica_matriz(a,b):
    """Multiplica duas matrizes 4x4 (a * b), recebidas e retornadas no formato
    achatado (1,16). Internamente usa np.dot após remodelar cada matriz para 4x4."""
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c

def translacao(tx, ty, tz):
    """Matriz de translação: desloca um ponto em (tx, ty, tz)."""
    return np.array([
        1, 0, 0, tx,
        0, 1, 0, ty,
        0, 0, 1, tz,
        0, 0, 0,  1
    ], np.float32)

def rotacao(rx, ry, rz):
    """Matriz de rotação combinada (Z * Y * X), com os ângulos `rx`, `ry` e `rz`
    (em radianos) representando a rotação em torno de cada eixo, respectivamente."""
    # Rotação em torno do eixo X
    c, s = np.cos(rx), np.sin(rx)
    X = np.array([
        1, 0,  0, 0,
        0, c, -s, 0,
        0, s,  c, 0,
        0, 0,  0, 1
    ], np.float32)

    # Rotação em torno do eixo Y
    c, s = np.cos(ry), np.sin(ry)
    Y = np.array([
         c, 0, s, 0,
         0, 1, 0, 0,
        -s, 0, c, 0,
         0, 0, 0, 1
    ], np.float32)

    # Rotação em torno do eixo Z
    c, s = np.cos(rz), np.sin(rz)
    Z = np.array([
        c, -s, 0, 0,
        s,  c, 0, 0,
        0,  0, 1, 0,
        0,  0, 0, 1
    ], np.float32)

    # Combina as três rotações na ordem Z * (Y * X)
    YX = multiplica_matriz(Y,X)
    return multiplica_matriz(Z,YX)

def escala(sx, sy, sz):
    """Matriz de escala não-uniforme, com fatores `sx`, `sy` e `sz` em cada eixo."""
    return np.array([
        sx,  0,  0, 0,
         0, sy,  0, 0,
         0,  0, sz, 0,
         0,  0,  0, 1
    ], np.float32)

In [9]:
glfw.show_window(window)

glEnable(GL_DEPTH_TEST)  # habilita o teste de profundidade (Z-buffer), essencial em 3D
                          # para que objetos mais próximos da câmera ocultem os mais distantes

# Centroide de todos os vértices da cena, usado como "pivô" para as rotações e o
# zoom globais da câmera, para que a cena gire/escale em torno do seu próprio centro
cx, cy, cz = vertices.mean(axis=0)

while not glfw.window_should_close(window):
    # Alterna entre modo preenchido (GL_FILL) e modo wireframe (GL_LINE) conforme a tecla P
    glPolygonMode(GL_FRONT_AND_BACK,GL_LINE if poly else GL_FILL)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(COR["fundo"][0], COR["fundo"][1], COR["fundo"][2], 1.0)
    

    # Atualiza os ângulos/escala acumulados com base nas velocidades atuais
    # (definidas/zeradas em key_event a cada tecla pressionada/solta)
    a_y += va_y
    a_x += va_x
    pos_peao = (peao_x, peao_z)
    a_torre += va_torre
    if ((2+sc > 0.7 and 2+sc < 3.5) or sc*vsc <= 0): sc += vsc # Limita o zoom

    # Calcula o deslocamento necessário para levar o peão controlável (que está
    # fisicamente fixo na casa (3,1) dentro do array de vértices) até a casa
    # (peao_x, peao_z) atual, apenas transladando-o
    t_x = grid[peao_x, peao_z][0] - grid[3, 1][0]
    t_z = grid[peao_x, peao_z][2] - grid[3, 1][2]

    # Acumula o tempo decorrido da animação de "rage quit", se estiver ativa
    if rage_quit:
        tempo_rage += dt

    # Desenha cada objeto da cena, calculando sua matriz de transformação específica
    for obj in objetos:
        if not obj.ativa:
            continue  # peças capturadas não são desenhadas

        # 1. Movimenta o peão principal (translação até a casa atual); os demais
        #    objetos começam com a matriz identidade (sem transformação própria)
        if obj.id == "peao":
            mat_transformation = translacao(t_x,0,t_z)
        else: 
            mat_transformation = np.identity(4, np.float32)

        # 2. Rotação da Torre branca da esquerda (Teclas C e V):
        #    move a peça para a origem, aplica a rotação em torno do eixo Y e
        #    devolve para sua posição original, fazendo-a girar sobre seu próprio eixo
        if obj.id == "torre_branca" and obj.pos == (0, 0):
            px, py, pz = grid[obj.pos]
            mat_transformation = multiplica_matriz(translacao(-px, -py, -pz), mat_transformation)
            mat_transformation = multiplica_matriz(rotacao(0, a_torre, 0), mat_transformation)
            mat_transformation = multiplica_matriz(translacao(px, py, pz), mat_transformation)

        # 3. Faz o zoom (aumenta de tamanho) na peça que está na mesma casa do peão,
        #    servindo como destaque visual da peça "alvo", a menos que o jogador
        #    tenha acabado de pressionar Enter, caso em que a peça é capturada
        #    (desativada) em vez de aumentada
        if obj.id not in ["tabuleiro", "peao"]:
            if pos_peao == obj.pos:
                if capturar:
                    obj.ativa = False
                    continue
                px, py, pz = grid[obj.pos]
                mat_transformation = multiplica_matriz(translacao(-px,-py,-pz), mat_transformation)
                mat_transformation = multiplica_matriz(escala(2,2,2), mat_transformation)
                mat_transformation = multiplica_matriz(translacao(px,py,pz), mat_transformation)

        # Animação de "Rage Quit": acionada pela tecla B, faz o tabuleiro tremer e
        # todas as peças voarem/girarem descontroladamente pelos ares
        if rage_quit:
            if obj.id == "tabuleiro":
                # Faz o tabuleiro tremer, com pequenos deslocamentos oscilantes em X e Z
                treme_x = np.sin(tempo_rage * 50) * 0.02
                treme_z = np.cos(tempo_rage * 45) * 0.02
                mat_transformation = multiplica_matriz(translacao(treme_x, 0, treme_z), mat_transformation)
            else:
                # Faz as peças voarem!
                px, py, pz = grid[obj.pos]
                
                # As peças vão para cima e para fora, com uma trajetória parabólica
                # simulando gravidade no eixo Y (sobem e depois caem)
                voo_x = px * tempo_rage * 3
                voo_y = (tempo_rage * 3) - (0.5 * 9.8 * (tempo_rage ** 2)) 
                voo_z = pz * tempo_rage * 3
                
                # Ângulo de giro descontrolado, crescente com o tempo, diferente em
                # cada eixo para um efeito mais caótico
                giro = tempo_rage * 15 
                
                # 1. Traz a peça pra origem, aplica o giro descontrolado e devolve
                mat_transformation = multiplica_matriz(translacao(-px, -py, -pz), mat_transformation)
                mat_transformation = multiplica_matriz(rotacao(giro, giro * 1.2, giro * 0.8), mat_transformation)
                mat_transformation = multiplica_matriz(translacao(px, py, pz), mat_transformation)
                
                # 2. Joga a peça pelos ares (translação de voo calculada acima)
                mat_transformation = multiplica_matriz(translacao(voo_x, voo_y, voo_z), mat_transformation)

        # 4. Aplica a escala e rotação globais da "Câmera": todas as transformações
        #    específicas do objeto (acima) já foram aplicadas; agora a cena inteira
        #    é recentrada na origem, escalada (zoom) e rotacionada em torno do seu
        #    centroide (cx, cy, cz), e então devolvida à posição original.
        #    -np.pi/8 é uma inclinação inicial fixa da câmera, somada à rotação
        #    controlada pelo usuário (a_x)
        mat_transformation = multiplica_matriz(translacao(-cx,-cy,-cz), mat_transformation)
        mat_transformation = multiplica_matriz(escala(2+sc,2+sc,2+sc), mat_transformation)
        mat_transformation = multiplica_matriz(rotacao(-np.pi/8+a_x,a_y,0), mat_transformation)
        mat_transformation = multiplica_matriz(translacao(cx,cy,cz), mat_transformation)
        
        # Envia a matriz de transformação final e a cor do objeto para os uniforms
        # do shader, e emite o comando de desenho dos triângulos deste objeto
        glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transformation) 
        glUniform4f(loc_color, obj.cor[0], obj.cor[1], obj.cor[2], 1.0)
        glDrawArrays(GL_TRIANGLES, obj.inicio, obj.quant)
        
    capturar = False  # a flag de captura vale só por 1 frame após o Enter
    
    glfw.swap_buffers(window)  # troca o buffer traseiro (recém-desenhado) com o visível na tela
    glfw.poll_events()         # processa eventos pendentes (teclado, fechar janela, etc.)

glfw.terminate()  # libera os recursos do GLFW ao sair do loop (janela fechada)